# Extract features with ECGFounder

In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd

BASE_DIR = os.path.dirname(os.path.abspath('.'))
RESULT_DIR = os.path.join(BASE_DIR, "results")
sys.path.append(BASE_DIR)

In [ ]:
from datautils import create_dataloader

ptbxl_loader = create_dataloader(f"{BASE_DIR}/data/cinc/ptbxl/test.h5",
                                 task="diagnostic_class", batch_size=32)
ptbxl_val_loader = create_dataloader(f"{BASE_DIR}/data/cinc/ptbxl/val.h5",
                                 task="diagnostic_class", batch_size=32)
print("Number of batches [32] in PTB-XL loader:", len(ptbxl_loader))

samitrop_loader = create_dataloader(f"{BASE_DIR}/data/cinc/samitrop/all_data.h5",
                                    task="normal_ecg", batch_size=32)
print("Number of batches [32] in SamiTrop loader:", len(samitrop_loader))

In [ ]:
ptbxl_val_labels = []
for _, labels in ptbxl_val_loader:
    ptbxl_val_labels.append(labels.numpy())
ptbxl_val_labels = np.concatenate(ptbxl_val_labels)

In [ ]:
ptbxl_labels = []
for _, labels in ptbxl_loader:
    ptbxl_labels.append(labels.numpy())
ptbxl_labels = np.concatenate(ptbxl_labels)

samitrop_labels = []
for _, labels in samitrop_loader:
    samitrop_labels.append(labels.numpy())
samitrop_labels = np.concatenate(samitrop_labels)

### Extract the features - ECGFounder

In [ ]:
from models import ft_12lead_ECGFounder
from utils import set_seed


device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")
set_seed(42, deterministic=True)

ecgfounder = ft_12lead_ECGFounder(pth=f"{BASE_DIR}/.checkpoints/12_lead_ECGFounder.pth",
                                  n_classes=2, linear_prob=True, device=device)
ecgfounder.return_features = True

In [ ]:
samitrop_features = []

ecgfounder.to(device)
ecgfounder.eval()

with torch.no_grad():

    for batch in samitrop_loader:
        x, y = batch[0].to(device), batch[1].to(device)

        _, deep_feat = ecgfounder(x)

        samitrop_features.append(deep_feat.detach().cpu())

ecgfounder.cpu()

samitrop_features = torch.concat(samitrop_features).numpy()
print("SamiTrop features shape:", samitrop_features.shape)

# np.save("samitrop_features.npy", samitrop_features)

In [ ]:
ptbxl_features = []

ecgfounder.to(device)
ecgfounder.eval()

with torch.no_grad():

    for batch in ptbxl_loader:
        x, y = batch[0].to(device), batch[1].to(device)

        _, deep_feat = ecgfounder(x)

        ptbxl_features.append(deep_feat.detach().cpu())

ecgfounder.cpu()

ptbxl_features = torch.concat(ptbxl_features).numpy()
print("PTB-XL features shape:", ptbxl_features.shape)

# np.save("ptbxl_test_features.npy", ptbxl_features)

In [ ]:
# ptbxl_val_features = []

# ecgfounder.to(device)
# ecgfounder.eval()

# with torch.no_grad():

#     for batch in ptbxl_val_loader:
#         x, y = batch[0].to(device), batch[1].to(device)

#         _, deep_feat = ecgfounder(x)

#         ptbxl_val_features.append(deep_feat.detach().cpu())

# ecgfounder.cpu()

# ptbxl_val_features = torch.concat(ptbxl_val_features).numpy()
# print("PTB-XL features shape:", ptbxl_val_features.shape)

# np.save("ptbxl_val_features.npy", ptbxl_val_features)

### Embeddings distribution analysis

In [ ]:
import plotly.express as ex
import plotly.io as pio
import plotly.graph_objects as go

# Set default template for all future figures
pio.templates["custom"] = go.layout.Template(
    layout=go.Layout(
        # xaxis=dict(showline=True, linewidth=1, linecolor='black', mirror=True),
        # yaxis=dict(showline=True, linewidth=1, linecolor='black', mirror=True),
        margin=dict(l=20, r=20, t=70, b=20)
    )
)
pio.templates.default = "plotly+custom"

In [ ]:
# Load the features from the .npy files
ptbxl_features = np.load("ptbxl_test_features.npy")
print("PTB-XL features shape:", ptbxl_features.shape)

# samitrop_features = np.load("samitrop_features.npy")
# print("SamiTrop features shape:", samitrop_features.shape)

# all_features = np.concatenate((ptbxl_features, samitrop_features), axis=0)
all_features = np.concatenate((ptbxl_features, ptbxl_val_features), axis=0)
print("All features shape:", all_features.shape)

# Create labels for the combined dataset 
# feat_dataset = np.concatenate((np.full(ptbxl_features.shape[0], "PTB-XL"), 
                            #    np.full(samitrop_features.shape[0], "SamiTrop")))
feat_dataset = np.concatenate((np.full(ptbxl_features.shape[0], "PTB-XL"), 
                               np.full(ptbxl_val_features.shape[0], "PTB-XL-val")))

#### UMAP

**UMAP Algorithm** - The major ones are as follows:

`n_neighbors`: This determines the number of neighboring points used in local approximations of manifold structure. Larger values will result in more global structure being preserved at the loss of detailed local structure. In general this parameter should often be in the range 5 to 50, with a choice of 10 to 15 being a sensible default.

`min_dist`: This controls how tightly the embedding is allowed compress points together. Larger values ensure embedded points are more evenly distributed, while smaller values allow the algorithm to optimise more accurately with regard to local structure. Sensible values are in the range 0.001 to 0.5, with 0.1 being a reasonable default.

`metric`: This determines the choice of metric used to measure distance in the input space. A wide variety of metrics are already coded, and a user defined function can be passed as long as it has been JITd by numba.

In [ ]:
import umap

# --- Perform UMAP dimensionality reduction (default parameters)
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
umap_features = reducer.fit_transform(all_features)

# np.savez_compressed("umap_features.npz", umap_features=umap_features, labels=feat_dataset)
np.savez_compressed("umap_ptbxl_features.npz", umap_features=umap_features, labels=feat_dataset)

In [ ]:
# --- Load the UMAP features and labels from the .npz file
# with np.load("umap_features.npz") as data:
#     umap_features = data["umap_features"]
#     feat_dataset = data["labels"]

_ptbxl_labels = np.where(ptbxl_labels[:,0] == 0, "ptbxl+abnormal", "ptbxl+normal")
# _samitrop_labels = np.where(samitrop_labels == 0, "samitrop+abnormal", "samitrop+normal")
_ptbxl_val_labels = np.where(ptbxl_val_labels[:,0] == 0, "ptbxl_val+abnormal", "ptbxl_val+normal")
_feat_labels = np.concatenate((_ptbxl_labels, _ptbxl_val_labels), axis=0)

In [ ]:
df_umap_feat = pd.DataFrame({
    "umap_dim1": umap_features[:, 0],
    "umap_dim2": umap_features[:, 1],
    "source": feat_dataset,
    "label": _feat_labels
})

In [ ]:
fig = ex.scatter(df_umap_feat, x="umap_dim1", y="umap_dim2", color="label", facet_col="source", facet_col_spacing=0.05)

fig.update_xaxes(title="Dimension 1", )
fig.update_yaxes(title="Dimension 2")
fig.update_layout(title="UMAP Projection of ECGFounder Features", title_y=0.97,
                  legend=dict(title="Label", orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1),
                  height=450, width=1000)
fig.show()

# fig.write_image(f"{RESULT_DIR}/ecgfounder_umap_feat_label.pdf")

In [ ]:
fig = ex.scatter(df_umap_feat, x='umap_dim1', y='umap_dim2', color='source')

fig.update_layout(title="UMAP Projection of ECGFounder Features",
                  xaxis_title="UMAP Dimension 1",
                  yaxis_title="UMAP Dimension 2",
                  title_y=0.97,
                  legend=dict(title="Source",orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=.5),
                  height=450, width=550)

fig.show()
# fig.write_image(f"{RESULT_DIR}/ecgfounder_umap_feat_source.pdf")

In [ ]:
fig = ex.scatter(x=umap_features[:,0], y=umap_features[:,1], color=_feat_labels)

fig.update_layout(title="UMAP Projection of ECGFounder Features",
                  xaxis_title="UMAP Dimension 1",
                  yaxis_title="UMAP Dimension 2",
                  title_y=0.97,
                  legend=dict(title="Class",orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=.5),
                  height=550, width=600)

fig.show()

#### CKA - Canonical Kernel Alignment

**Demo code for "[Similarity of Neural Network Representations Revisited](https://arxiv.org/abs/1905.00414)"**

Copyright 2019 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

Please cite as:

    @inproceedings{pmlr-v97-kornblith19a,
      title = {Similarity of Neural Network Representations Revisited},
      author = {Kornblith, Simon and Norouzi, Mohammad and Lee, Honglak and Hinton, Geoffrey},
      booktitle = {Proceedings of the 36th International Conference on Machine Learning},
      pages = {3519--3529},
      year = {2019},
      volume = {97},
      month = {09--15 Jun},
      publisher = {PMLR}
    }

In [ ]:
import numpy as np

def gram_linear(x):
  """Compute Gram (kernel) matrix for a linear kernel.

  Args:
    x: A num_examples x num_features matrix of features.

  Returns:
    A num_examples x num_examples Gram matrix of examples.
  """
  return x.dot(x.T)


def gram_rbf(x, threshold=1.0):
  """Compute Gram (kernel) matrix for an RBF kernel.

  Args:
    x: A num_examples x num_features matrix of features.
    threshold: Fraction of median Euclidean distance to use as RBF kernel
      bandwidth. (This is the heuristic we use in the paper. There are other
      possible ways to set the bandwidth; we didn't try them.)

  Returns:
    A num_examples x num_examples Gram matrix of examples.
  """
  dot_products = x.dot(x.T)
  sq_norms = np.diag(dot_products)
  sq_distances = -2 * dot_products + sq_norms[:, None] + sq_norms[None, :]
  sq_median_distance = np.median(sq_distances)
  return np.exp(-sq_distances / (2 * threshold ** 2 * sq_median_distance))


def center_gram(gram, unbiased=False):
  """Center a symmetric Gram matrix.

  This is equvialent to centering the (possibly infinite-dimensional) features
  induced by the kernel before computing the Gram matrix.

  Args:
    gram: A num_examples x num_examples symmetric matrix.
    unbiased: Whether to adjust the Gram matrix in order to compute an unbiased
      estimate of HSIC. Note that this estimator may be negative.

  Returns:
    A symmetric matrix with centered columns and rows.
  """
  if not np.allclose(gram, gram.T):
    raise ValueError('Input must be a symmetric matrix.')
  gram = gram.copy()

  if unbiased:
    # This formulation of the U-statistic, from Szekely, G. J., & Rizzo, M.
    # L. (2014). Partial distance correlation with methods for dissimilarities.
    # The Annals of Statistics, 42(6), 2382-2412, seems to be more numerically
    # stable than the alternative from Song et al. (2007).
    n = gram.shape[0]
    np.fill_diagonal(gram, 0)
    means = np.sum(gram, 0, dtype=np.float64) / (n - 2)
    means -= np.sum(means) / (2 * (n - 1))
    gram -= means[:, None]
    gram -= means[None, :]
    np.fill_diagonal(gram, 0)
  else:
    means = np.mean(gram, 0, dtype=np.float64)
    means -= np.mean(means) / 2
    gram -= means[:, None]
    gram -= means[None, :]

  return gram


def cka(gram_x, gram_y, debiased=False):
  """Compute CKA.

  Args:
    gram_x: A num_examples x num_examples Gram matrix.
    gram_y: A num_examples x num_examples Gram matrix.
    debiased: Use unbiased estimator of HSIC. CKA may still be biased.

  Returns:
    The value of CKA between X and Y.
  """
  gram_x = center_gram(gram_x, unbiased=debiased)
  gram_y = center_gram(gram_y, unbiased=debiased)

  # Note: To obtain HSIC, this should be divided by (n-1)**2 (biased variant) or
  # n*(n-3) (unbiased variant), but this cancels for CKA.
  scaled_hsic = gram_x.ravel().dot(gram_y.ravel())

  normalization_x = np.linalg.norm(gram_x)
  normalization_y = np.linalg.norm(gram_y)
  return scaled_hsic / (normalization_x * normalization_y)


def _debiased_dot_product_similarity_helper(
    xty, sum_squared_rows_x, sum_squared_rows_y, squared_norm_x, squared_norm_y,
    n):
  """Helper for computing debiased dot product similarity (i.e. linear HSIC)."""
  # This formula can be derived by manipulating the unbiased estimator from
  # Song et al. (2007).
  return (
      xty - n / (n - 2.) * sum_squared_rows_x.dot(sum_squared_rows_y)
      + squared_norm_x * squared_norm_y / ((n - 1) * (n - 2)))


def feature_space_linear_cka(features_x, features_y, debiased=False):
  """Compute CKA with a linear kernel, in feature space.

  This is typically faster than computing the Gram matrix when there are fewer
  features than examples.

  Args:
    features_x: A num_examples x num_features matrix of features.
    features_y: A num_examples x num_features matrix of features.
    debiased: Use unbiased estimator of dot product similarity. CKA may still be
      biased. Note that this estimator may be negative.

  Returns:
    The value of CKA between X and Y.
  """
  features_x = features_x - np.mean(features_x, 0, keepdims=True)
  features_y = features_y - np.mean(features_y, 0, keepdims=True)

  dot_product_similarity = np.linalg.norm(features_x.T.dot(features_y)) ** 2
  normalization_x = np.linalg.norm(features_x.T.dot(features_x))
  normalization_y = np.linalg.norm(features_y.T.dot(features_y))

  if debiased:
    n = features_x.shape[0]
    # Equivalent to np.sum(features_x ** 2, 1) but avoids an intermediate array.
    sum_squared_rows_x = np.einsum('ij,ij->i', features_x, features_x)
    sum_squared_rows_y = np.einsum('ij,ij->i', features_y, features_y)
    squared_norm_x = np.sum(sum_squared_rows_x)
    squared_norm_y = np.sum(sum_squared_rows_y)
  
    dot_product_similarity = _debiased_dot_product_similarity_helper(
        dot_product_similarity, sum_squared_rows_x, sum_squared_rows_y,
        squared_norm_x, squared_norm_y, n)
    normalization_x = np.sqrt(_debiased_dot_product_similarity_helper(
        normalization_x ** 2, sum_squared_rows_x, sum_squared_rows_x,
        squared_norm_x, squared_norm_x, n))
    normalization_y = np.sqrt(_debiased_dot_product_similarity_helper(
        normalization_y ** 2, sum_squared_rows_y, sum_squared_rows_y,
        squared_norm_y, squared_norm_y, n))

  print(dot_product_similarity, normalization_x, normalization_y, normalization_x*normalization_y)

  return dot_product_similarity / (normalization_x * normalization_y)

In [ ]:
# --- Linear CKA ---
def linear_cka(X, Y):
    """
    X: (n, d1)
    Y: (n, d2)
    """
    # Center features
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)

    print(np.abs(X.mean()), np.abs(Y.mean()))

    # Compute dot products
    dot_product_similarity = np.linalg.norm(X.T @ Y) ** 2
    normalization_x = np.linalg.norm(X.T @ X)
    normalization_y = np.linalg.norm(Y.T @ Y)

    return dot_product_similarity / (normalization_x * normalization_y)


# --- RBF kernel and kernel CKA ---
def rbf_kernel(X, sigma=None):
    GX = X @ X.T
    KX = np.diag(GX)[:, None] - 2 * GX + np.diag(GX)[None, :]

    if sigma is None:
        # median heuristic
        sigma = np.median(KX[KX > 0])

    KX = np.exp(-KX / (2 * sigma))
    return KX

def center_gram(K):
    """Center a Gram matrix."""
    n = K.shape[0]
    unit = np.ones((n, n)) / n
    return K - unit @ K - K @ unit + unit @ K @ unit

def kernel_cka(X, Y, sigma=None):
    K = center_gram(rbf_kernel(X, sigma))
    L = center_gram(rbf_kernel(Y, sigma))

    hsic = np.sum(K * L)
    norm_k = np.sqrt(np.sum(K * K))
    norm_l = np.sqrt(np.sum(L * L))

    return hsic / (norm_k * norm_l)

In [ ]:
ptbxl_x = np.concatenate([x for x, _ in ptbxl_loader])
ptbxl_val_x = np.concatenate([x for x, _ in ptbxl_val_loader])

In [ ]:
Z1 = ptbxl_features
Z2 = ptbxl_val_features # samitrop_features

In [ ]:
Z1.shape

In [ ]:
Z1 = ptbxl_features
Z2 = ptbxl_val_features # samitrop_features

# Ensure same number of samples
n = min(len(Z1), len(Z2))
Z1a = Z1[:500,]
Z1b = Z1[500:1000,]

# cka()

# Z1 = Z1[:n]
# Z2 = Z2[:n]
# print("!Selected only the first", n, "samples from each dataset for CKA computation.\n")


lin_cka_score = linear_cka(Z1a, Z1b)
print("Linear CKA similarity:", lin_cka_score)
# lin_cka_score = feature_space_linear_cka(Z1, Z2)
# print("Linear CKA similarity:", lin_cka_score)



# ker_cka_score = kernel_cka(Z1, Z2)
# print("Kernel CKA (RBF) similarity:", ker_cka_score)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

S = cosine_similarity(Z1)

In [ ]:
Z1.shape

In [ ]:
fig = ex.imshow(S, aspect='auto', origin='lower')

fig.update_layout(title=f"Cosine Similarity [Z1]",
                  xaxis_title="Samples",
                  yaxis_title="Samples",
                  height=550, width=600)
fig.show()

In [ ]:
print(np.linalg.norm(Z1), np.linalg.norm(Z2))

In [ ]:
print(np.linalg.norm(Z1.T @ Z2))

In [ ]:
Z1 = ptbxl_features
Z2 = ptbxl_val_features # samitrop_features


# Ensure same number of samples
n = min(len(Z1), len(Z2))
Z1 = Z1[:n]
Z2 = Z2[:n]

# Center
Z1 = Z1 - Z1.mean(0)
Z2 = Z2 - Z2.mean(0)

# Reduce noise
k = min(50, Z1.shape[1], Z2.shape[1])
Z1 = PCA(k).fit_transform(Z1)
Z2 = PCA(k).fit_transform(Z2)

# Compute CKA
cka_val = np.linalg.norm(Z1.T @ Z2, 'fro')**2
cka_val /= (
    np.linalg.norm(Z1.T @ Z1, 'fro') *
    np.linalg.norm(Z2.T @ Z2, 'fro')
)

print(cka_val)

In [ ]:
np.random.seed(42)
X = np.random.randn(2183, 1024)
Y = np.random.randn(2183, 1024) + X

cka_from_examples = cka(gram_linear(X), gram_linear(Y))
print("Linear CKA similarity:", cka_from_examples)

#### PCA+CKA

In [ ]:
from sklearn.decomposition import PCA

def pca_cka_analysis(Z1, Z2, k=50, cka="linear"):
    # Center
    Z1 = Z1 - Z1.mean(axis=0)
    Z2 = Z2 - Z2.mean(axis=0)

    # PCA
    pca1 = PCA(n_components=k)
    pca2 = PCA(n_components=k)

    Z1_pca = pca1.fit_transform(Z1)
    Z2_pca = pca2.fit_transform(Z2)

    # Cross-component similarity
    C = Z1_pca.T @ Z2_pca

    # Normalize to cosine similarity
    norms1 = np.linalg.norm(Z1_pca, axis=0)
    norms2 = np.linalg.norm(Z2_pca, axis=0)

    C = C / norms1[:, None]
    C = C / norms2[None, :]

    # CKA
    if cka == "linear":
        cka_score = linear_cka(Z1_pca, Z2_pca)
    elif cka == "rbf":
        cka_score = kernel_cka(Z1_pca, Z2_pca)
    else:
        raise ValueError("Invalid CKA type. Choose 'linear' or 'rbf'.")

    return {
        "C": C,                      # component (cosine) similarity matrix
        "cka": cka_score,                  # global similarity
        "var1": pca1.explained_variance_ratio_,
        "var2": pca2.explained_variance_ratio_,
    }

In [ ]:
report = pca_cka_analysis(Z1, Z2, k=50, cka="rbf")
print("PCA-CKA similarity:", report["cka"])

In [ ]:
df_var_ratio = pd.DataFrame({"ratio": np.concatenate((report["var1"], report["var2"])),
                            "dataset": ["ptbxl"]*len(report["var1"]) + ["ptbxl-val"]*len(report["var2"]),
                            "pca_component": np.concatenate((np.arange(len(report["var1"]))+1, np.arange(len(report["var2"]))+1))})

In [ ]:
fig = ex.histogram(df_var_ratio, y='ratio', x='pca_component', facet_col='dataset', nbins=50, 
                   title=f"PCA Variance Ratio [PCA-CKA = {report['cka']:.3f}]", facet_col_spacing=0.08)
fig.update_yaxes(title="Proportion of residual variance")
fig.update_xaxes(title="PCA Component")
fig.update_layout(font_size=13, height=350, width=1200)
fig.show()
# fig.write_image(f"{RESULT_DIR}/ecgfounder_umap_pca_var.pdf")

In [ ]:
report["C"].shape

In [ ]:
fig = ex.imshow(report["C"], aspect='auto', origin='lower')
fig.update_layout(title=f"PCA Cosine Similarity [PCA-CKA = {report['cka']:.3f}]",
                  xaxis_title="SamiTrop Components",
                  yaxis_title="PTB-XL Components",
                  title_y=0.97,
                  height=550, width=600)
fig.show()
# fig.write_image(f"{RESULT_DIR}/ecgfounder_pca_similarity.pdf")